# Keyword & Top People EDA

**ASSIGNED: MARK**

Jasmin had done a general analysis on the dataset during the first EDA. I added in 2 tabs to do additional analysis for top keywords, bigrams, and people. 

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import networkx as nx
from networkx.algorithms import community

from collections import Counter
from itertools import combinations
from IPython.display import display  

In [ ]:
df = pd.read_parquet("../data_storage/processed_data/sampled_data.parquet")

## General Analysis

In [ ]:
def build_tokens(df: pd.DataFrame, body_col: str = "article_body") -> pd.DataFrame:
    if body_col not in df.columns:
        raise KeyError(f"Column '{body_col}' not found in DataFrame.")
    out = df.copy()
    out["tokens"] = out[body_col].fillna("").apply(preprocess_article_text_nltk)
    out["keyword_counts"] = out["tokens"].apply(lambda toks: Counter(toks))
    return out

df = build_tokens(df, body_col="article_body")

In [ ]:
### keyword frequency table 
def keyword_frequency_table(df: pd.DataFrame, tokens_col: str = "tokens") -> pd.DataFrame:
    if tokens_col not in df.columns:
        raise KeyError(f"Column '{tokens_col}' not found in DataFrame.")
    kw = (df.explode(tokens_col)
            .groupby(tokens_col)
            .size()
            .reset_index(name="frequency")
            .sort_values("frequency", ascending=False)
            .rename(columns={tokens_col: "keyword"}))
    return kw

keywords_df = keyword_frequency_table(df)
display(keywords_df.head(25))

In [ ]:
### keyword co-occurence list
def keyword_cooccurrence_edges(df: pd.DataFrame, tokens_col: str = "tokens", min_co: int = 2) -> pd.DataFrame:
    """
    Builds edges between keywords that co-occur in the same article.
    Each article contributes at most one count per keyword pair (set() ensures uniqueness per article).
    """
    rows = []
    for toks in df[tokens_col].dropna():
        uniq = sorted(set(toks))
        rows.extend((a, b) for a, b in combinations(uniq, 2))
    if not rows:
        return pd.DataFrame(columns=["kw1", "kw2", "weight"])
    edges = (pd.DataFrame(rows, columns=["kw1", "kw2"])
               .value_counts()
               .reset_index(name="weight")
               .sort_values("weight", ascending=False))
    # prune weak edges
    edges = edges[edges["weight"] >= min_co].reset_index(drop=True)
    return edges

edges_kw = keyword_cooccurrence_edges(df, tokens_col="tokens", min_co=2)
display(edges_kw.head(25))
print("keywords:", keywords_df.shape[0], "| co-occurrence edges:", edges_kw.shape[0])

In [ ]:
### building keyword graphs 
def build_keyword_graph(edges_kw: pd.DataFrame, min_weight: int = 2) -> nx.Graph:
    """
    edges_kw columns: ['kw1','kw2','weight']
    Creates an undirected weighted graph of keyword co-occurrence.
    """
    required = {"kw1","kw2","weight"}
    if not required.issubset(edges_kw.columns):
        raise ValueError(f"edges_kw must have columns {required}")
    df = edges_kw.copy()
    df = df[df["weight"] >= min_weight]
    G = nx.Graph()
    G.add_weighted_edges_from(df[["kw1","kw2","weight"]].itertuples(index=False, name=None))
    return G

In [ ]:
### visualizing the graphs 
def visualize_keyword_graph(
    G: nx.Graph,
    title: str = "Keyword Co-occurrence Network",
    label_top: int = 30,
    edge_keep_pct: int = 70,      # keep strongest X% edges by weight
    top_nodes_by: str = "strength" # "strength" (weighted degree) | "degree"
):
    if G.number_of_nodes() == 0:
        print("Graph is empty."); return

    # keep strongest edges by percentile
    w = np.array([G[u][v].get("weight", 1) for u, v in G.edges()])
    thr = np.percentile(w, 100 - edge_keep_pct) if len(w) else 1
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from([(u, v, d) for u, v, d in G.edges(data=True) if d.get("weight", 1) >= thr])
    H.remove_nodes_from([n for n in list(H.nodes) if H.degree(n) == 0])
    if H.number_of_nodes() == 0:
        print("No nodes after pruning."); return

    # largest connected component
    comps = list(nx.connected_components(H))
    H = H.subgraph(max(comps, key=len)).copy()

    # node sizes by weighted degree (“strength”)
    strength = {n: sum(H[n][nbr].get("weight", 1) for nbr in H.neighbors(n)) for n in H.nodes()}
    degree = dict(H.degree())
    size_map = {n: 120 + 14*np.log1p(strength.get(n, 1)) for n in H.nodes()}
    ewidths = [0.6 + np.log1p(H[u][v].get("weight", 1)) for u, v in H.edges()]

    # layout
    pos = nx.spring_layout(H, seed=42, k=1/np.sqrt(max(1, H.number_of_nodes())))

    plt.figure(figsize=(12, 9))
    nx.draw_networkx_edges(H, pos, width=ewidths, alpha=0.35)
    nx.draw_networkx_nodes(H, pos, node_size=[size_map[n] for n in H.nodes()])

    # labels: top nodes by chosen metric
    if top_nodes_by == "degree":
        rank = sorted(degree.items(), key=lambda x: x[1], reverse=True)
    else:
        rank = sorted(strength.items(), key=lambda x: x[1], reverse=True)
    top_nodes = {n for n, _ in rank[:label_top]}
    nx.draw_networkx_labels(H, pos, labels={n: n for n in top_nodes}, font_size=9)

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
### Top Edges, Top Keywords by strength
def keyword_lists(G: nx.Graph, edges_kw: pd.DataFrame, top_n: int = 25):
    # strongest edges (by weight)
    edge_table = (edges_kw.sort_values("weight", ascending=False)
                           .head(top_n)
                           .reset_index(drop=True))

    # node strength (weighted degree)
    strength = pd.Series({n: sum(G[n][nbr].get("weight",1) for nbr in G.neighbors(n)) for n in G.nodes()}, name="strength")
    top_strength = strength.sort_values(ascending=False).head(top_n).to_frame()

    # classic centralities (unweighted degree & betweenness on giant component)
    if G.number_of_nodes() > 0 and G.number_of_edges() > 0:
        comps = list(nx.connected_components(G))
        H = G.subgraph(max(comps, key=len)).copy()
        deg = pd.Series(dict(H.degree()), name="degree").sort_values(ascending=False).head(top_n).to_frame()
        btw = pd.Series(nx.betweenness_centrality(H, normalized=True), name="betweenness") \
                .sort_values(ascending=False).head(top_n).to_frame()
    else:
        deg = pd.DataFrame(columns=["degree"])
        btw = pd.DataFrame(columns=["betweenness"])

    return edge_table, top_strength, deg, btw

In [ ]:
### color-coding communities
def keyword_communities(G: nx.Graph, min_size: int = 5):
    if G.number_of_nodes() == 0:
        return {}
    # greedy modularity communities (no extra deps)
    comms = list(community.greedy_modularity_communities(G))
    # keep only communities with at least min_size
    return {i: sorted(list(c)) for i, c in enumerate(c for c in comms if len(c) >= min_size)}

In [ ]:
G_kw = build_keyword_graph(edges_kw, min_weight=2)
print("keywords:", G_kw.number_of_nodes(), "| edges:", G_kw.number_of_edges())

# 2) Visualize (tune these like you did for Source→Topic)
visualize_keyword_graph(
    G_kw,
    title="Keyword Co-occurrence (min_weight=2, top edges kept)",
    label_top=30,
    edge_keep_pct=75,         # keep strongest 75% edges → prune 25% weakest
    top_nodes_by="strength"   # or "degree"
)

# 3) Lists (tables) like before
edge_table, top_strength, deg_table, btw_table = keyword_lists(G_kw, edges_kw, top_n=25)
display(edge_table.head(25))      # strongest keyword pairs
display(top_strength.head(25))    # most influential keywords by weighted degree
display(deg_table.head(25))       # highest-degree keywords
display(btw_table.head(25))       # highest-betweenness keywords

# 4) (Optional) Communities
comms = keyword_communities(G_kw, min_size=5)
for cid, members in comms.items():
    print(f"Community {cid} (n={len(members)}):", ", ".join(members[:15]), "...")

## Top Words

In [ ]:
keywords = pd.read_parquet("../data_storage/final_data/top_1000_keywords.parquet")

In [ ]:
bigrams = pd.read_parquet("../data_storage/final_data/top_1000_bigrams.parquet")

## Top People

In [ ]:
people = pd.read_parquet("../data_storage/final_data/persons_detected.parquet")

In [ ]:
people_row = pd.read_parquet("../data_storage/final_data/persons_by_row.parquet")